# Serviço de Recomendação

Demonstração do serviço de decisão em operação.

A política Thompson Sampling é inicializada com o estado aprendido a partir do histórico da camada _trusted_: para cada canal, o parâmetro α acumula as conversões observadas e β as não conversões, somados ao prior não informativo Beta(1,1).

In [1]:
# Carga de dados

import pandas as pd
import numpy as np

df = pd.read_parquet('../data/trusted/bank_clean.parquet')
ARMS = ['cellular', 'telephone']

# Estado aprendido a partir do histórico
alfa = {arm: 1 + int(df.loc[df['contact'] == arm, 'y'].sum()) for arm in ARMS}
beta = {arm: 1 + int((df['contact'] == arm).sum()) - (alfa[arm] - 1) for arm in ARMS}

for arm in ARMS:
    print(f"{arm}: alfa={alfa[arm]}, beta={beta[arm]}, "
          f"taxa estimada={alfa[arm]/(alfa[arm]+beta[arm]):.4f}")

cellular: alfa=4370, beta=24917, taxa estimada=0.1492
telephone: alfa=391, beta=2517, taxa estimada=0.1345


O serviço expõe duas operações:

- `recomendar(cliente)` — recebe os dados de um cliente elegível e devolve o   canal de contato recomendado, acompanhado da taxa estimada para aquele   canal e dos valores sorteados de cada distribuição.
- `registrar_feedback(canal, converteu)` — recebe o resultado observado após o contato e atualiza o estado da política.

A segunda operação é o que caracteriza uma plataforma adaptativa: a decisão não é congelada em um modelo treinado offline, mas ajustada continuamente a partir das respostas observadas. As células finais demonstram esse ciclo, comparando a recomendação antes e depois do registro de uma conversão.

Em produção, essas funções corresponderiam aos endpoints de uma API, com o estado persistido em banco de baixa latência, conforme a arquitetura descrita no README.

In [2]:
rng = np.random.default_rng()

def recomendar(cliente: dict) -> dict:
    """Recebe os dados de um cliente e devolve o canal recomendado."""
    sorteios = {arm: rng.beta(alfa[arm], beta[arm]) for arm in ARMS}
    escolhido = max(sorteios, key=sorteios.get)

    return {
        'cliente': cliente,
        'canal_recomendado': escolhido,
        'taxa_estimada': round(alfa[escolhido] / (alfa[escolhido] + beta[escolhido]), 4),
        'sorteios': {k: round(v, 4) for k, v in sorteios.items()},
    }


def registrar_feedback(canal: str, converteu: int) -> None:
    """Atualiza o estado da política com o resultado observado."""
    if converteu == 1:
        alfa[canal] += 1
    else:
        beta[canal] += 1
    print(f"{canal}: alfa={alfa[canal]}, beta={beta[canal]}")

In [3]:
#Dados do cliente

cliente = {
    'age': 58,
    'job': 'retired',
    'marital': 'married',
    'education': 'secondary',
    'balance': 2100,
    'housing': 'no',
    'loan': 'no',
}

resposta = recomendar(cliente)
resposta

{'cliente': {'age': 58,
  'job': 'retired',
  'marital': 'married',
  'education': 'secondary',
  'balance': 2100,
  'housing': 'no',
  'loan': 'no'},
 'canal_recomendado': 'cellular',
 'taxa_estimada': 0.1492,
 'sorteios': {'cellular': 0.1512, 'telephone': 0.1267}}

In [4]:
# O contato foi feito e o cliente converteu
registrar_feedback(resposta['canal_recomendado'], converteu=1)

# Nova recomendação, já com o estado atualizado
recomendar(cliente)

cellular: alfa=4371, beta=24917


{'cliente': {'age': 58,
  'job': 'retired',
  'marital': 'married',
  'education': 'secondary',
  'balance': 2100,
  'housing': 'no',
  'loan': 'no'},
 'canal_recomendado': 'cellular',
 'taxa_estimada': 0.1492,
 'sorteios': {'cellular': 0.149, 'telephone': 0.1337}}

## Demonstração de mudança de comportamento

Para validar a mudança de comportamento em função do aprendizado, serão feitas 300 alimentações de chamadas observadas no telefone que permitirá avaliar que o aumento de recomendação de canal telefone para contato.

**Observação:** esta célula modifica o estado em memória. Executá-la
repetidamente sem reiniciar o kernel acumula o efeito.

In [5]:
# Demonstração do aprendizado: 300 conversões observadas no telefone
for _ in range(300):
    registrar_feedback('telephone', converteu=1)

print("\nEstado após o feedback:")
for arm in ARMS:
    print(f"{arm}: taxa estimada={alfa[arm]/(alfa[arm]+beta[arm]):.4f}")

print("\nDistribuição das recomendações em 100 chamadas:")
escolhas = [recomendar(cliente)['canal_recomendado'] for _ in range(100)]
print(pd.Series(escolhas).value_counts())

telephone: alfa=392, beta=2517
telephone: alfa=393, beta=2517
telephone: alfa=394, beta=2517
telephone: alfa=395, beta=2517
telephone: alfa=396, beta=2517
telephone: alfa=397, beta=2517
telephone: alfa=398, beta=2517
telephone: alfa=399, beta=2517
telephone: alfa=400, beta=2517
telephone: alfa=401, beta=2517
telephone: alfa=402, beta=2517
telephone: alfa=403, beta=2517
telephone: alfa=404, beta=2517
telephone: alfa=405, beta=2517
telephone: alfa=406, beta=2517
telephone: alfa=407, beta=2517
telephone: alfa=408, beta=2517
telephone: alfa=409, beta=2517
telephone: alfa=410, beta=2517
telephone: alfa=411, beta=2517
telephone: alfa=412, beta=2517
telephone: alfa=413, beta=2517
telephone: alfa=414, beta=2517
telephone: alfa=415, beta=2517
telephone: alfa=416, beta=2517
telephone: alfa=417, beta=2517
telephone: alfa=418, beta=2517
telephone: alfa=419, beta=2517
telephone: alfa=420, beta=2517
telephone: alfa=421, beta=2517
telephone: alfa=422, beta=2517
telephone: alfa=423, beta=2517
telephon